In [1]:
from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from transformers import get_cosine_schedule_with_warmup
drive.mount('/content/gdrive', force_remount=True)


base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2"
# base_dir = "/content/gdrive/MyDrive/Final Project"
sys.path.append(base_dir)

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


Mounted at /content/gdrive
Device set to cuda


In [2]:
import Datasets.DataLoader as DataLoader_Lib


from Models.BERT_Model import BERT_Lag, BERTConfig
from Datasets.DataLoader import CombinedBinDataLoader

In [3]:
batch_per_iter = 32 # Adjust batch size based on your Colab GPU memory
grad_acc_factor = 16
eff_batch_size = batch_per_iter * grad_acc_factor
block_size = 512
warmup_steps = 150
num_steps_train = 3623
num_steps_val = 10
weight_decay = .1
dropout = 0.1
lr = 3e-4
min_lr = 3e-5
tokens_per_step = eff_batch_size * block_size

config = BERTConfig(num_heads = 12,
  num_layers = 12,
  vocab_size = 50257,
  embedding_dim = 768,
  block_size = block_size,
  dropout = dropout,
  weight_decay = weight_decay,
  pad_token_id=50256)

model = BERT_Lag(config, device)
model = model.to(device)
model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(),
                              betas=(0.9, 0.95),
                              lr=lr,
                              weight_decay=weight_decay)

def get_lr(step):
    # warmup
    if step < warmup_steps:
        return lr * step / warmup_steps
    # cosine decay to min_lr
    progress = (step - warmup_steps) / (num_steps_train - warmup_steps)
    cosine   = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + cosine * (lr - min_lr)

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda step: get_lr(step) / lr
)
torch.set_float32_matmul_precision('high')

scaler = torch.amp.GradScaler('cuda')

In [4]:
!cp "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/Datasets/fineweb_1B.bin" "/content/fineweb_1B.bin"
train_loader, val_loader = CombinedBinDataLoader.create_loaders(
    '/content/fineweb_1B.bin', batch_per_iter, block_size, config, seed=0
)

Initialized loader with 57,983 chunks of size 16385.
Initialized loader with 3,052 chunks of size 16385.


In [5]:
@torch.no_grad()
def estimate_loss(model, loader, device, eval_iters=10, lam=.5):
    model.eval()

    losses_fwd = []
    losses_bwd = []


    for i in range(eval_iters):
        x, y, _ = loader.get_data()
        x, y = x.to(device), y.to(device)
        loss_fwd, loss_bwd = model(x, y)
        losses_fwd.append(loss_fwd.item())
        losses_bwd.append(loss_bwd.item())
        del x, y, loss_fwd, loss_bwd

    avg_fwd = sum(losses_fwd) / len(losses_fwd)
    ppl_fwd = torch.exp(torch.tensor(avg_fwd)).item()
    avg_bwd = sum(losses_bwd) / len(losses_bwd)
    ppl_bwd = torch.exp(torch.tensor(avg_bwd)).item()

    model.train()
    return avg_fwd, avg_bwd, ppl_fwd, ppl_bwd



def train_loop(model, optimizer, scheduler, device, train_loader, val_loader,
               num_steps_train, num_steps_val, lam = .5):

    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    history = {
        'train_steps': [],
        'train_loss':  [],
        'val_steps':   [],
        'val_loss_fwd':[],
        'val_loss_bwd':[],
        'val_ppl_fwd': [],
        'val_ppl_bwd': []
    }
    loss_fwd, loss_bwd, ppl_fwd, ppl_bwd = estimate_loss(model, val_loader, device, num_steps_val)
    history['val_steps'].append(0)
    history['val_loss_fwd'].append(loss_fwd)
    history['val_ppl_fwd'].append(ppl_fwd)
    history['val_loss_bwd'].append(loss_bwd)
    history['val_ppl_bwd'].append(ppl_bwd)

    print(f"Step    0 | Val FWD: {loss_fwd:.4f} | PPL_FWD: {ppl_fwd:.2f} | Val BWD: {loss_bwd:.4f}| PPL_BWD: {ppl_bwd:.2f}")
    start = time.time()
    tokens_seen = 0

    #Define some useful constants

    for step in range(num_steps_train):
        if step % 50 == 0 and step > 0:
            if device == 'mps':
                torch.mps.empty_cache()
            elif device == 'cuda':
              torch.cuda.empty_cache()
              torch.cuda.ipc_collect()
            loss_fwd, loss_bwd, ppl_fwd, ppl_bwd = estimate_loss(model, val_loader, device, num_steps_val)

            history['val_steps'].append(step)
            history['val_loss_fwd'].append(loss_fwd)
            history['val_ppl_fwd'].append(ppl_fwd)
            history['val_loss_bwd'].append(loss_bwd)
            history['val_ppl_bwd'].append(ppl_bwd)

            print(f"Step  {step}| Val FWD: {loss_fwd:.4f} | PPL_FWD: {ppl_fwd:.2f} | Val BWD: {loss_bwd:.4f}| PPL_BWD: {ppl_bwd:.2f}")


        avg_loss = 0
        optimizer.zero_grad(set_to_none=True)
        for i in range(grad_acc_factor):
            x, y, _ = train_loader.get_data()
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)


            with torch.autocast(device_type=device.type, dtype=torch.float16):
                loss_fwd, loss_bwd = model(x, y)
                loss_fwd /= grad_acc_factor
                loss_bwd /= grad_acc_factor

                step_loss = lam * loss_fwd + (1- lam) * loss_bwd
            scaler.scale(step_loss).backward()
            avg_loss += step_loss.detach().item()  # ← scalar for logging
            del x, y, loss_fwd, loss_bwd, step_loss

        scaler.unscale_(optimizer)          # unscale before clipping
        clip_grad_norm_(model.parameters(), max_norm=2.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        tokens_seen += tokens_per_step
        if step % 10 == 0:
            stop = time.time()
            history['train_steps'].append(step)
            history['train_loss'].append(avg_loss)
            print(f"step {step:4d}/{num_steps_train} | tokens {tokens_seen:,} | Train loss {avg_loss:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | Time Since Last Train Print {(stop-start):.4f} seconds")
            start = time.time()
    return history


In [ ]:
torch.cuda.empty_cache()
history = train_loop(model, optimizer, scheduler, device, train_loader, val_loader, num_steps_train, grad_acc_factor * 2)

Trainable parameters: 124,083,456


W0422 03:47:13.932000 20846 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Step    0 | Val FWD: 10.9868 | PPL_FWD: 59087.82 | Val BWD: 10.9857| PPL_BWD: 59026.21
step    0/3623 | tokens 262,144 | Train loss 10.9869 | lr 2.00e-06 | Time Since Last Train Print 71.2075 seconds
step   10/3623 | tokens 2,883,584 | Train loss 10.3883 | lr 2.20e-05 | Time Since Last Train Print 122.8510 seconds
step   20/3623 | tokens 5,505,024 | Train loss 9.6560 | lr 4.20e-05 | Time Since Last Train Print 126.4302 seconds
step   30/3623 | tokens 8,126,464 | Train loss 9.1677 | lr 6.20e-05 | Time Since Last Train Print 124.7992 seconds
step   40/3623 | tokens 10,747,904 | Train loss 8.6786 | lr 8.20e-05 | Time Since Last Train Print 123.8065 seconds
Step  50| Val FWD: 8.2073 | PPL_FWD: 3667.78 | Val BWD: 8.2068| PPL_BWD: 3665.81
step   50/3623 | tokens 13,369,344 | Train loss 8.2354 | lr 1.02e-04 | Time Since Last Train Print 138.8657 seconds
step   60/3623 | tokens 15,990,784 | Train loss 7.9217 | lr 1.22e-04 | Time Since Last Train Print 122.1472 seconds
step   70/3623 | tokens 1

In [ ]:
loss_fwd, loss_bwd, ppl_fwd, ppl_bwd = estimate_loss(model, val_loader, device, grad_acc_factor)
print(f"Val FWD: {loss_fwd:.4f} | PPL FWD: {ppl_fwd:.2f} | PPL BWD: {ppl_bwd:.2f} | Val BWD: {-100:.4f}")

In [ ]:
path = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/ckpt_BERT_6_layers_combined_loss.pt"
torch.save({
            "step": 2770,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict()
        }, path)

In [ ]:
import tiktoken

def generate(model, prompt, max_new_tokens=200, temperature=1.0, top_k=50, device=device):
    enc = tiktoken.get_encoding("gpt2")
    tokens = enc.encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)  # [1, T]

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            x_cond = x[:, -model.config.block_size:]

            B, N = x_cond.size()
            pos = torch.arange(0, N, dtype=torch.long, device=device)

            with torch.amp.autocast('cuda'):
                h = model.transformer.drop(
                    model.transformer.wte(x_cond) + model.transformer.wpe(pos)
                )
                for block in model.transformer.h:
                    h = block(h, mask=True)
                h = model.transformer.ln_f(h)
                logits = model.lm_head(h)  # [1, T, vocab_size]

            logits = logits[:, -1, :].float() / temperature  # [1, vocab_size]

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            x = torch.cat([x, next_token], dim=1)

    model.train()
    return enc.decode(x[0].tolist())

print(generate(model, """Ancient Rome (753 BC–AD 476) evolved from a small Italian city-state into a massive Mediterranean empire, structured into three main periods: the Monarchy/Kingdom (753–509 BC), the Republic (509–27 BC), and the Empire (27 BC–AD 476).
It became a dominant power under emperors like Augustus, falling in the West due to internal instability and external pressures, while the Eastern Byzantine Empire continued until 1453.
""", max_new_tokens=100, temperature=1, top_k=50))